# 02 XGBoost Regressor (Macro)

Predict next-quarter real GDP growth with gradient boosting. Compare against OLS and Random Forest from notebooks `00` and `01`.

## Table of Contents
- [Why gradient boosting](#why)
- [Environment bootstrap](#bootstrap)
- [Load data + split](#load-split)
- [Fit XGBoost with early stopping](#fit-early-stop)
- [Hyperparameter search via walk-forward CV](#tune)
- [Evaluate the tuned model](#evaluate)
- [Predicted vs actual + residuals](#diagnostics)
- [Feature importance: gain vs permutation](#importance)
- [Final comparison: OLS vs RF vs XGBoost](#compare)
- [Reflection](#reflection)

<a id="why"></a>
## Why Gradient Boosting

Random Forest fits independent trees in parallel and averages them. **XGBoost** fits trees sequentially: each new tree corrects the residual errors of the ensemble built so far. The two approaches reduce error in different ways:

- Random Forest reduces **variance** by averaging.
- Gradient boosting reduces **bias** by adding many weak learners that each chip away at the residuals.

On tabular data — including macro panels — gradient boosting often wins, but it is more sensitive to hyperparameters. The two knobs to think about together are `learning_rate` and `n_estimators`: cutting `learning_rate` in half roughly doubles the trees you need, but produces a smoother decision surface that generalizes better.

**Early stopping** is the safest way to pick `n_estimators`: train on a training set, watch the validation log-loss / RMSE, and stop adding trees when it has not improved in (say) 50 rounds.

<a id="bootstrap"></a>
## Environment Bootstrap

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time


def find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(8):
        if (p / 'src').exists() and (p / 'docs').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root.')


PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SAMPLE_DIR = PROJECT_ROOT / 'data' / 'sample'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
print('Repo root:', PROJECT_ROOT)

<a id="load-split"></a>
## Load Data + Split

Same data, same target, same split as notebooks `00` and `01`.

In [ ]:
import numpy as np
import pandas as pd
from src.evaluation import time_train_test_split_index, regression_metrics

path = PROCESSED_DIR / 'macro_quarterly.csv'
if path.exists():
    df = pd.read_csv(path, index_col=0, parse_dates=True)
else:
    df = pd.read_csv(SAMPLE_DIR / 'macro_quarterly_sample.csv', index_col=0, parse_dates=True)

y_col = 'gdp_growth_qoq'
x_cols = ['T10Y2Y_lag1', 'UNRATE_lag1', 'FEDFUNDS_lag1', 'INDPRO_lag1', 'RSAFS_lag1']
data = df[[y_col] + x_cols].dropna().copy()

split = time_train_test_split_index(len(data), test_size=0.2)
train, test = data.iloc[split.train_slice], data.iloc[split.test_slice]
X_train, y_train = train[x_cols].to_numpy(), train[y_col].to_numpy()
X_test, y_test = test[x_cols].to_numpy(), test[y_col].to_numpy()
print(f'Train: {len(train)}   Test: {len(test)}')

<a id="fit-early-stop"></a>
## Fit XGBoost with Early Stopping

We split the *training* set into an inner train (80%) and validation slice (20%) and let XGBoost stop when the validation RMSE stops improving.

In [ ]:
from xgboost import XGBRegressor

n_inner = int(0.8 * len(X_train))
X_inner, y_inner = X_train[:n_inner], y_train[:n_inner]
X_val, y_val = X_train[n_inner:], y_train[n_inner:]

xgb_es = XGBRegressor(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.9,
    tree_method='hist',
    early_stopping_rounds=50,
    eval_metric='rmse',
    random_state=0,
)
xgb_es.fit(X_inner, y_inner, eval_set=[(X_val, y_val)], verbose=False)

print(f'Best iteration: {xgb_es.best_iteration}')
yhat_es = xgb_es.predict(X_test)
es_metrics = regression_metrics(y_test, yhat_es)
es_metrics

<a id="tune"></a>
## Hyperparameter Search via Walk-Forward CV

Boosting is sensitive to depth × learning_rate. We hold `n_estimators=500` (with early stopping inside each fold) and search over (`max_depth`, `learning_rate`).

In [ ]:
from itertools import product
from src.evaluation import walk_forward_splits

X_full = data[x_cols].to_numpy()
y_full = data[y_col].to_numpy()
n = len(data)

grid = {
    'max_depth': [3, 4, 6],
    'learning_rate': [0.03, 0.05, 0.1],
}

results = []
for max_depth, lr in product(grid['max_depth'], grid['learning_rate']):
    fold_rmse = []
    for sp in walk_forward_splits(n, initial_train_size=int(0.5 * n), test_size=8, step_size=8):
        Xtr_full, ytr_full = X_full[sp.train_slice], y_full[sp.train_slice]
        n_inner = int(0.85 * len(Xtr_full))
        m = XGBRegressor(
            n_estimators=1000, max_depth=max_depth, learning_rate=lr,
            subsample=0.8, colsample_bytree=0.9, tree_method='hist',
            early_stopping_rounds=30, eval_metric='rmse', random_state=0,
        )
        m.fit(Xtr_full[:n_inner], ytr_full[:n_inner],
              eval_set=[(Xtr_full[n_inner:], ytr_full[n_inner:])], verbose=False)
        yp = m.predict(X_full[sp.test_slice])
        fold_rmse.append(regression_metrics(y_full[sp.test_slice], yp)['rmse'])
    results.append({'max_depth': max_depth, 'learning_rate': lr,
                    'cv_rmse_mean': float(np.mean(fold_rmse)),
                    'cv_rmse_std': float(np.std(fold_rmse))})

results_df = pd.DataFrame(results).sort_values('cv_rmse_mean').reset_index(drop=True)
results_df

<a id="evaluate"></a>
## Evaluate the Tuned Model

In [ ]:
best = results_df.iloc[0]
print(f'Best params: max_depth={int(best["max_depth"])}, learning_rate={best["learning_rate"]}')

n_inner = int(0.85 * len(X_train))
xgb_tuned = XGBRegressor(
    n_estimators=1500,
    max_depth=int(best['max_depth']),
    learning_rate=float(best['learning_rate']),
    subsample=0.8,
    colsample_bytree=0.9,
    tree_method='hist',
    early_stopping_rounds=50,
    eval_metric='rmse',
    random_state=0,
)
t0 = time.time()
xgb_tuned.fit(X_train[:n_inner], y_train[:n_inner],
              eval_set=[(X_train[n_inner:], y_train[n_inner:])], verbose=False)
xgb_train_secs = time.time() - t0
yhat_tuned = xgb_tuned.predict(X_test)
tuned_metrics = regression_metrics(y_test, yhat_tuned)
print(f'Trained in {xgb_train_secs:.2f}s with {xgb_tuned.best_iteration} trees')
tuned_metrics

<a id="diagnostics"></a>
## Predicted vs Actual + Residuals

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(test.index, y_test, label='Actual', color='black', linewidth=1.5)
axes[0].plot(test.index, yhat_tuned, label='XGBoost tuned', color='darkorange', linewidth=1.5)
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].set_ylabel('GDP growth (QoQ %)')
axes[0].set_title('Predicted vs actual on the held-out test set')
axes[0].legend()
axes[1].plot(test.index, y_test - yhat_tuned, color='steelblue')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('Residual')
axes[1].set_xlabel('Quarter')
axes[1].set_title('Residuals over time')
plt.tight_layout()
plt.show()

<a id="importance"></a>
## Feature Importance: Gain vs Permutation

XGBoost reports its built-in importance using `gain` (the average improvement in loss the feature produces when used in a split). This is biased the same way Random Forest's built-in importance is. As before, **permutation importance** on the test set is the honest version.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(xgb_tuned, X_test, y_test, n_repeats=30, random_state=0, n_jobs=-1)
imp_df = pd.DataFrame({
    'feature': x_cols,
    'permutation_importance_mean': perm.importances_mean,
    'permutation_importance_std': perm.importances_std,
    'gain_importance': xgb_tuned.feature_importances_,
}).sort_values('permutation_importance_mean', ascending=False)
imp_df

<a id="compare"></a>
## Final Comparison: OLS vs Random Forest vs XGBoost

All three models on the same data, same split, same metric. The right answer here depends on the dataset — sometimes OLS wins, sometimes XGBoost — and that's the point of running them side by side.

In [ ]:
from src.econometrics import fit_ols, fit_ols_hac
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm

ols = fit_ols(train, y_col=y_col, x_cols=x_cols)
X_test_design = sm.add_constant(test[x_cols], has_constant='add')
yhat_ols = ols.predict(X_test_design).to_numpy()
ols_metrics = regression_metrics(y_test, yhat_ols)

t0 = time.time()
rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=3, random_state=0, n_jobs=-1)
rf.fit(X_train, y_train)
rf_train_secs = time.time() - t0
yhat_rf = rf.predict(X_test)
rf_metrics = regression_metrics(y_test, yhat_rf)

summary = pd.DataFrame({
    'OLS':            {**ols_metrics, 'train_secs': 0.0},
    'RandomForest':   {**rf_metrics,  'train_secs': round(rf_train_secs, 3)},
    'XGBoost (tuned)': {**tuned_metrics, 'train_secs': round(xgb_train_secs, 3)},
})
summary

<a id="reflection"></a>
## Reflection

1. Which model wins on RMSE? On R²? On training time? Most production systems care about all three — sketch the tradeoff curve in your head.
2. The OLS model gives you confidence intervals on each coefficient (you saw HAC SEs in `00`). The tree models do not. If a policymaker asked "is the unemployment-rate effect statistically distinguishable from zero?" — which model would you turn to, and why?
3. With ~150 quarters of synthetic data, the tree models often *fail* to beat OLS. Macroeconomic relationships tend to be approximately linear and the sample is small. Pick a real-world prediction task you can imagine where you would expect XGBoost to beat OLS by a wide margin — what features of that task make it favorable to boosting?